In [0]:
from pyspark.sql.functions import when, col, lit, sum as spark_sum, count, log, exp

## Rates per TPMA

In [0]:
rates_data = spark.read.parquet("/Volumes/nhp/inputs_data/files/dev/lad23cd/rates.parquet")
display(rates_data)

In [0]:
national_rates_data = (
    rates_data
    .filter((col("lad23cd") == "national") & col("fyear").between(201516, 202324))
    .withColumn("numerator", col("crude_rate") * col("denominator"))
    .drop("lad23cd")
)

display(national_rates_data)

## Totals per activity type and by mitigation

### apc


#### Note about sample rates

If a given `epikey` is in a set of strategies s_i for i = 1, ..., n, and those strategies have sample rates 0 < sr_i <= 1, the combined sample rate is:

$$
1 - \prod_{i=1}^{n} (1 - sr_i)
$$


In [0]:
apc = spark.table("nhp.raw_data.apc")
apc_mitig = spark.table("nhp.raw_data.apc_mitigators")

In [0]:
# Fast path: epikeys with at least one deterministic strategy (most records)
deterministic = (
    apc_mitig
    .filter(col("sample_rate") == 1.0)
    .select("fyear", "provider", "epikey")
    .distinct()
    .withColumn("combined_sample_rate", lit(1.0))
)

# Slow path: epikeys ONLY in non-deterministic strategies (rare)
non_deterministic = (
    apc_mitig
    .join(deterministic.select("fyear", "provider", "epikey"), 
          on=["fyear", "provider", "epikey"], 
          how="left_anti")  # Exclude epikeys already in deterministic
    .groupBy("fyear", "provider", "epikey")
    .agg(
        (lit(1.0) - exp(spark_sum(log(lit(1.0) - col("sample_rate"))))).alias("combined_sample_rate")
    )
)

# Combine both
apc_mitig_combined = deterministic.unionByName(non_deterministic)

display(apc_mitig_combined)

#### Getting the summary counts

In [0]:
apc_join_cols = ["epikey", "fyear", "provider"]

apc = apc.filter(col("fyear").between(201516, 202324))

# 1. MITIGABLE counts: Sum of combined_sample_rate for joined episodes
apc_joined_mitig = (
    apc.join(
        apc_mitig_combined.select(*apc_join_cols, "combined_sample_rate").distinct(),
        on=apc_join_cols,
        how="inner"
    )
    .groupBy("fyear")
    .agg(
        spark_sum(col("combined_sample_rate")).alias("count")
    )
    .withColumn("mitigable", lit("mitigable"))
)

# 2. USUAL counts from joined episodes: Sum of (1 - combined_sample_rate)
apc_joined_usual = (
    apc.join(
        apc_mitig_combined.select(*apc_join_cols, "combined_sample_rate").distinct(),
        on=apc_join_cols,
        how="inner"
    )
    .groupBy("fyear")
    .agg(
        spark_sum(lit(1) - col("combined_sample_rate")).alias("count")
    )
    .withColumn("mitigable", lit("usual"))
)

# 3. USUAL counts from non-joined episodes: Count of anti-joined records
apc_not_joined_usual = (
    apc.join(
        apc_mitig_combined.select(*apc_join_cols).distinct(),
        on=apc_join_cols,
        how="left_anti"
    )
    .groupBy("fyear")
    .agg(
        count("*").alias("count")
    )
    .withColumn("mitigable", lit("usual"))
)

# 4. Combine all USUAL counts
apc_usual_combined = (
    apc_joined_usual
    .unionByName(apc_not_joined_usual)
    .groupBy("fyear", "mitigable")
    .agg(spark_sum("count").alias("count"))
)

# 5. TOTAL counts for verification (independent calculation)
apc_total = (
    apc
    .groupBy("fyear")
    .agg(count("*").alias("count"))
    .withColumn("mitigable", lit("all"))
)

# 6. Combine all categories
apc_counts = (
    apc_joined_mitig
    .unionByName(apc_usual_combined)
    .unionByName(apc_total)
    .orderBy("fyear", "mitigable")
)

display(apc_counts)

# Optional: Verification check
# This should show that mitigable + usual = all for each fyear
verification = (
    apc_counts
    .groupBy("fyear")
    .pivot("mitigable")
    .agg(spark_sum("count"))
    .withColumn("check", col("mitigable") + col("usual") - col("all"))
)
display(verification)

### opa

In [0]:
opa = spark.table("nhp.raw_data.opa")
opa_mitig = spark.table("nhp.raw_data.opa_mitigators")

In [0]:
opa_join_cols = ["attendkey", "fyear", "provider"]

opa = opa.filter(col("fyear").between(201516, 202324))

opa_mitig = opa.join(
    opa_mitig.select(*opa_join_cols).distinct(),
    on=opa_join_cols,
    how="left_semi"
).withColumn("mitigable", lit("mitigable"))

opa_usual = opa.join(
    opa_mitig.select(*opa_join_cols).distinct(),
    on=opa_join_cols,
    how="left_anti"
).withColumn("mitigable", lit("usual"))

opa_by_mitig = opa_mitig.unionByName(opa_usual)

opa_total = opa.withColumn("mitigable", lit("all"))

opa_final = opa_by_mitig.unionByName(opa_total)

opa_counts = (
    opa_final
    .groupBy("fyear", "mitigable")
    .agg(count("attendkey")))


display(opa_counts)

## ED

In [0]:
ed_data = spark.table("nhp.raw_data.ecds")

In [0]:
ed_data = ed_data.filter(col("fyear").between(201516, 202324))

# Potential filter to type 1 departments
# ed_data = ed_data.filter(col("aedepttype") == "01")

ed_mitig = (
    ed_data
    .withColumn(
        "mitigable",
        when(
            (col("is_frequent_attender")) |
            (col("is_left_before_treatment")) |
            (col("is_low_cost_referred_or_discharged")) |
            (col("is_discharged_no_treatment") ),
            "mitigable"
        ).otherwise("usual")
    )
)

ed_all = ed_data.withColumn("mitigable", lit("all"))

ed_final = ed_mitig.unionByName(ed_all)

ed_counts = ed_final.groupBy("fyear", "mitigable").count()

display(ed_counts)